# Shareholder Yield Factor Index — Russell 1000 Universe

This notebook constructs a factor-tilted index using two signals:

1. **Shareholder Yield (level)** — dividends + net buybacks, scaled by market cap
2. **Shareholder Yield Growth (trend)** — YoY change in total shareholder payout

The universe is the Russell 1000. All data is pulled through the **LSEG Data Library for Python** (`lseg-data`).

## Methodology Summary

**Shareholder Yield:**

$$SY_i = \frac{|D_i| + \max(|B_i| - |S_i|,\; 0)}{M_i}$$

where $D_i$ is dividends paid, $B_i$ is share repurchases, $S_i$ is share issuance, and $M_i$ is market capitalization.

**Shareholder Yield Growth** (measured on the numerator to avoid price-denominator noise):

$$g_i = \frac{\text{Payout}_{i,t} - \text{Payout}_{i,t-1}}{\text{Payout}_{i,t-1}}$$

**Composite score** (after cross-sectional z-scoring):

$$C_i = w_1 \cdot z_{i,SY} + w_2 \cdot z_{i,gSY}$$

**Tilt-based weighting** against the market-cap benchmark:

$$w_i^{index} = w_i^{mktcap} \cdot e^{\lambda C_i}$$

(then renormalize, apply position caps, and apply sector caps)

## Prerequisites

```bash
pip install lseg-data pandas numpy scipy matplotlib
```

You also need a valid `lseg-data.config.json` configured for either a Desktop Session (requires Workspace or Eikon running locally) or a Platform Session. Russell 1000 constituent data is fee-liable — check your entitlements.

## 1. Imports and Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import time
import logging
from datetime import datetime, timedelta
from typing import Optional

import lseg.data as ld
import pandas as pd
import numpy as np
from scipy.stats.mstats import winsorize
import matplotlib.pyplot as plt

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(message)s")
log = logging.getLogger(__name__)

%matplotlib inline

## 2. Configuration

All methodology knobs live in the `Config` class. Changing signal weights, tilt strength, sector caps, or rebalance frequency happens here and nowhere else. This makes experimentation much cleaner — you can tweak one parameter and rerun the pipeline without hunting through the code.

In [ ]:
class Config:
    """Central configuration for the index methodology."""

    # Universe
    INDEX_RIC = ".RUI"                          # Russell 1000
    BENCHMARK_RIC = ".RUI"                      # Benchmark for performance comparison

    # Signal parameters
    LOOKBACK_YEARS = 5                          # Years of history for the backtest
    SY_GROWTH_LOOKBACK_MONTHS = 12              # YoY window for shareholder yield growth

    # Scoring
    WINSORIZE_LIMITS = (0.025, 0.025)           # 2.5th / 97.5th percentile clip
    SIGNAL_WEIGHTS = {                          # Composite signal weights
        "sy_level": 0.50,
        "sy_growth": 0.50,
    }

    # Portfolio construction
    WEIGHTING_METHOD = "tilt"                   # "tilt" | "score" | "top_n"
    TOP_N = 200                                 # Used only if WEIGHTING_METHOD == "top_n"
    TILT_LAMBDA = 2.0                           # Exponential tilt strength
    MIN_WEIGHT = 0.0005                         # Floor weight (5 bps)
    MAX_WEIGHT = 0.05                           # Cap weight at 5%
    SECTOR_CAP_MULTIPLE = 2.0                   # Max sector weight = benchmark × this

    # Rebalancing
    REBALANCE_FREQ = "Q"                        # Q = quarterly
    BUFFER_RANK = 50                            # Hysteresis buffer for top-N method

    # Transaction cost model
    COST_PER_SIDE_BPS = 15                      # One-way cost in basis points

    # LSEG API batching
    API_BATCH_SIZE = 80                         # Max instruments per get_data call
    API_SLEEP_SECONDS = 1.5                     # Throttle between API calls

## 3. Open LSEG Session

This opens a session using the credentials in your `lseg-data.config.json`. If you have Workspace running locally, the default Desktop Session should just work.

In [ ]:
ld.open_session()
log.info("LSEG session opened.")

## 4. Get Russell 1000 Constituents

Russell constituent data is fee-liable on LSEG and sometimes gated behind a separate entitlement. The function below tries three methods in order:

1. `TR.IndexConstituentRIC` — the cleanest path if you have the right permissions
2. Screener-based lookup — falls back to the equity screener expression
3. Chain RIC (`0#.RUI`) — last resort

If all three fail, your entitlements likely don't include Russell index data — contact your LSEG account team.

In [ ]:
def get_constituents(index_ric: str, as_of_date: Optional[str] = None) -> list:
    """
    Retrieve Russell 1000 constituent RICs.

    Parameters
    ----------
    index_ric : str
        The index RIC, e.g. ".RUI"
    as_of_date : str, optional
        Date string "YYYYMMDD" for point-in-time constituents.

    Returns
    -------
    list[str]
        List of constituent RICs.
    """
    params = {}
    if as_of_date:
        params["SDate"] = as_of_date

    # Method 1: TR.IndexConstituentRIC
    try:
        df = ld.get_data(
            universe=index_ric,
            fields=["TR.IndexConstituentRIC"],
            parameters=params,
        )
        rics = df["Constituent RIC"].dropna().tolist()
        if len(rics) > 100:
            log.info(f"Retrieved {len(rics)} constituents via TR.IndexConstituentRIC.")
            return rics
    except Exception as e:
        log.warning(f"TR.IndexConstituentRIC failed: {e}")

    # Method 2: Screener fallback
    try:
        screener_expr = (
            'SCREEN(U(IN(Equity(active,public,primary))),'
            'IN(TR.IndexName,"Russell 1000"),'
            'CURN=USD)'
        )
        df = ld.get_data(screener_expr, ["TR.RIC"])
        rics = df.iloc[:, -1].dropna().tolist()
        log.info(f"Retrieved {len(rics)} constituents via Screener fallback.")
        return rics
    except Exception as e:
        log.warning(f"Screener fallback failed: {e}")

    # Method 3: Chain RIC
    try:
        chain_ric = f"0#{index_ric}"
        df = ld.get_data(chain_ric, ["TR.RIC"])
        rics = df.iloc[:, -1].dropna().tolist()
        log.info(f"Retrieved {len(rics)} constituents via chain RIC.")
        return rics
    except Exception as e:
        raise RuntimeError(
            f"Could not retrieve constituents for {index_ric}. "
            "Check your LSEG entitlements — Russell index data may require "
            "a separate data license."
        ) from e


rics = get_constituents(Config.INDEX_RIC)
print(f"Universe size: {len(rics)} stocks")
print(f"First 10 RICs: {rics[:10]}")

## 5. Fetch Fundamental Data

We pull the raw inputs from the cash flow statement for two fiscal years:

- **FY0** (most recent completed fiscal year) — for the level signal
- **FY-1** (prior fiscal year) — for the growth signal

The LSEG TR field codes used:

| Field | Meaning |
|-------|---------|
| `TR.F.DivsPd` | Total cash dividends paid (cash outflow, reported as negative) |
| `TR.F.ComStkBuyback` | Cash spent on share repurchases (negative) |
| `TR.F.ComStkIssd` | Cash received from share issuance (positive) |
| `TR.CompanyMarketCap` | Market capitalization |
| `TR.TRBCEconomicSector` | Sector classification |

We batch the API calls to respect LSEG rate limits (default: 80 instruments per call, with a short sleep between batches).

In [ ]:
def fetch_fundamental_data(
    rics: list,
    fields: list,
    parameters: dict,
    batch_size: int = Config.API_BATCH_SIZE,
) -> pd.DataFrame:
    """Fetch fundamental data in batches to respect LSEG API limits."""
    frames = []
    total_batches = (len(rics) + batch_size - 1) // batch_size

    for i in range(0, len(rics), batch_size):
        batch = rics[i : i + batch_size]
        batch_num = i // batch_size + 1
        log.info(f"  Fetching batch {batch_num}/{total_batches} ({len(batch)} instruments)...")

        try:
            df = ld.get_data(universe=batch, fields=fields, parameters=parameters)
            frames.append(df)
        except Exception as e:
            log.warning(f"  Batch {batch_num} failed: {e}")

        if batch_num < total_batches:
            time.sleep(Config.API_SLEEP_SECONDS)

    if not frames:
        raise RuntimeError("All data batches failed. Check session and entitlements.")

    return pd.concat(frames, ignore_index=True)


def fetch_shareholder_yield_inputs(rics: list, period: str) -> pd.DataFrame:
    """Retrieve raw inputs needed to compute shareholder yield for one period."""
    fields = [
        "TR.F.DivsPd",
        "TR.F.ComStkBuyback",
        "TR.F.ComStkIssd",
        "TR.CompanyMarketCap",
        "TR.TRBCEconomicSector",
        "TR.PriceClose",
        "TR.F.DivsPd.periodenddate",
    ]
    params = {"Period": period, "Curn": "USD", "Scale": "6"}  # Scale=6 → millions
    return fetch_fundamental_data(rics, fields, params)

In [ ]:
# Fetch current fiscal year
log.info("Fetching shareholder yield data (current year = FY0)...")
df_fy0 = fetch_shareholder_yield_inputs(rics, period="FY0")
df_fy0.head()

In [ ]:
# Fetch prior fiscal year
log.info("Fetching shareholder yield data (prior year = FY-1)...")
df_fy1 = fetch_shareholder_yield_inputs(rics, period="FY-1")
df_fy1.head()

### Inspect Returned Columns

LSEG's TR field codes return display names that can vary slightly by entitlement. It's worth inspecting the columns here — if the names below don't match what's in the signal functions (like "Dividends Paid", "Common Stock Bought Back"), update the column references in the next section.

In [ ]:
print("FY0 columns:", df_fy0.columns.tolist())
print()
print("Data types:")
print(df_fy0.dtypes)

## 6. Signal Construction

### Shareholder Yield (Level)

$$SY_i = \frac{|D_i| + \max(|B_i| - |S_i|,\; 0)}{M_i}$$

Dividends and buybacks are reported as negative values in the cash flow statement (cash outflows), so we take absolute values. Issuance is a positive cash inflow, so net buyback is $|B_i| - |S_i|$, floored at zero so that net issuers don't get negative credit. This matches how most commercial shareholder yield indices handle the issuance question.

### Shareholder Yield Growth

Measured on the **numerator** (total payout in dollars), not the yield ratio:

$$g_i = \frac{\text{Payout}_{i,t} - \text{Payout}_{i,t-1}}{\text{Payout}_{i,t-1}}$$

This isolates management's capital allocation decision from stock price noise. If we measured yield-ratio growth instead, a 30% price drop would register as yield "growth" even if the company's actual payout policy didn't change.

In [ ]:
def _total_payout(df: pd.DataFrame) -> pd.Series:
    """Compute total dollar payout (dividends + net buybacks, floored at 0)."""
    divs = df["Dividends Paid"].fillna(0).abs()
    buybacks = df["Common Stock Bought Back"].fillna(0).abs()
    issuance = df["Common Stock Issued"].fillna(0).abs()
    return divs + (buybacks - issuance).clip(lower=0)


def compute_shareholder_yield(df: pd.DataFrame) -> pd.Series:
    """Compute shareholder yield = total payout / market cap."""
    payout = _total_payout(df)
    mktcap = df["Company Market Cap"]
    sy = payout / mktcap
    return sy.replace([np.inf, -np.inf], np.nan)


def compute_shareholder_yield_growth(
    payout_current: pd.Series, payout_prior: pd.Series
) -> pd.Series:
    """Compute YoY growth in the payout numerator (not the yield ratio)."""
    growth = (payout_current - payout_prior) / payout_prior.replace(0, np.nan)
    return growth.replace([np.inf, -np.inf], np.nan)

In [ ]:
# Set RIC as index (first column from get_data is typically the instrument)
ric_col = df_fy0.columns[0]
df_fy0 = df_fy0.set_index(ric_col)
df_fy1 = df_fy1.set_index(ric_col)

# Compute raw signals
sy_current = compute_shareholder_yield(df_fy0)
payout_current = _total_payout(df_fy0)
payout_prior = _total_payout(df_fy1)
sy_growth = compute_shareholder_yield_growth(payout_current, payout_prior)

# Quick look at signal distributions
signal_df = pd.DataFrame({
    "shareholder_yield": sy_current,
    "sy_growth": sy_growth,
})
print("Signal Summary Statistics:")
print(signal_df.describe())

### Visualize Signal Distributions

Before z-scoring, let's look at the raw distributions. Expect to see heavy right tails on shareholder yield (a few companies with very high payouts) and potentially wild outliers on growth (where last year's payout was tiny). That's exactly why we winsorize before z-scoring.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Clip for display purposes (not for actual calculations)
sy_display = sy_current.clip(-0.05, 0.25)
growth_display = sy_growth.clip(-2, 5)

axes[0].hist(sy_display.dropna(), bins=50, color="#2563eb", edgecolor="white")
axes[0].set_title("Shareholder Yield Distribution")
axes[0].set_xlabel("Shareholder Yield")
axes[0].set_ylabel("Count")
axes[0].axvline(sy_current.median(), color="red", linestyle="--", label=f"Median: {sy_current.median():.2%}")
axes[0].legend()

axes[1].hist(growth_display.dropna(), bins=50, color="#059669", edgecolor="white")
axes[1].set_title("Shareholder Yield Growth Distribution")
axes[1].set_xlabel("YoY Payout Growth")
axes[1].set_ylabel("Count")
axes[1].axvline(sy_growth.median(), color="red", linestyle="--", label=f"Median: {sy_growth.median():.2%}")
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Cross-Sectional Z-Scoring and Composite Score

Winsorize at the 2.5th/97.5th percentiles, then z-score:

$$z_{i,k} = \frac{x_{i,k} - \bar{x}_k}{\sigma_k}$$

Combine into a composite:

$$C_i = w_{SY} \cdot z_{i,SY} + w_{gSY} \cdot z_{i,gSY}$$

With default `SIGNAL_WEIGHTS = {"sy_level": 0.50, "sy_growth": 0.50}`, we're weighting the level and growth signals equally. Adjust in the Config class to tilt one way or the other.

In [ ]:
def cross_sectional_zscore(signal: pd.Series, limits: tuple = Config.WINSORIZE_LIMITS) -> pd.Series:
    """Winsorize then z-score a cross-sectional signal."""
    valid = signal.dropna()
    if len(valid) < 10:
        return signal * np.nan

    winsorized = pd.Series(
        winsorize(valid.values, limits=limits),
        index=valid.index,
        dtype=float,
    )
    mu = winsorized.mean()
    sigma = winsorized.std()
    if sigma < 1e-12:
        return winsorized * 0.0

    z = (winsorized - mu) / sigma
    return z.reindex(signal.index)


def build_composite_score(
    z_sy: pd.Series, z_sy_growth: pd.Series, weights: dict = Config.SIGNAL_WEIGHTS
) -> pd.Series:
    """Combine z-scored signals into a single composite."""
    composite = (
        weights["sy_level"] * z_sy.fillna(0)
        + weights["sy_growth"] * z_sy_growth.fillna(0)
    )
    # Penalize stocks missing both signals
    both_missing = z_sy.isna() & z_sy_growth.isna()
    composite[both_missing] = np.nan
    return composite

In [ ]:
z_sy = cross_sectional_zscore(sy_current)
z_sy_growth = cross_sectional_zscore(sy_growth)
composite = build_composite_score(z_sy, z_sy_growth)

print(f"Signal coverage:")
print(f"  SY level:    {z_sy.notna().sum()} stocks")
print(f"  SY growth:   {z_sy_growth.notna().sum()} stocks")
print(f"  Composite:   {composite.notna().sum()} stocks")

# Signal correlation — useful sanity check
print("\nSignal correlation matrix:")
print(pd.DataFrame({"z_sy": z_sy, "z_sy_growth": z_sy_growth}).corr().round(3))

### Top and Bottom Composite Scores

Let's eyeball the names — are the top scorers companies you'd expect to see as generous capital returners? Are the bottom scorers high-growth or financially stressed names?

In [ ]:
top_10 = composite.nlargest(10)
bottom_10 = composite.nsmallest(10)

print("Top 10 Composite Scores (high shareholder yield & growth):")
for ric, score in top_10.items():
    sy_val = sy_current.get(ric, np.nan)
    g_val = sy_growth.get(ric, np.nan)
    print(f"  {ric:<12}  Composite: {score:+.2f}   SY: {sy_val:.2%}   Growth: {g_val:+.2%}")

print("\nBottom 10 Composite Scores:")
for ric, score in bottom_10.items():
    sy_val = sy_current.get(ric, np.nan)
    g_val = sy_growth.get(ric, np.nan)
    print(f"  {ric:<12}  Composite: {score:+.2f}   SY: {sy_val:.2%}   Growth: {g_val:+.2%}")

## 8. Portfolio Construction

Three weighting methods available (set via `Config.WEIGHTING_METHOD`):

**`"tilt"`** — Start from market-cap weights, tilt exponentially by composite score:

$$w_i^{index} = w_i^{mktcap} \cdot e^{\lambda C_i}$$

This keeps the portfolio close to the benchmark and limits tracking error. Controlled by `TILT_LAMBDA` (higher = more aggressive).

**`"score"`** — Weight purely by composite score (floored at zero). More aggressive factor exposure but higher tracking error.

**`"top_n"`** — Select top N stocks, then weight by market cap within that subset.

After initial weighting, we apply:
- **Single-name caps**: `MIN_WEIGHT` / `MAX_WEIGHT`
- **Sector caps**: No sector can be more than `SECTOR_CAP_MULTIPLE × benchmark_sector_weight`

In [ ]:
def compute_index_weights(
    composite: pd.Series,
    mktcap: pd.Series,
    sectors: pd.Series,
    method: str = Config.WEIGHTING_METHOD,
) -> pd.Series:
    """Convert composite scores into index weights."""
    # Drop stocks with missing data
    valid = composite.dropna().index.intersection(mktcap.dropna().index)
    comp = composite.loc[valid]
    mc = mktcap.loc[valid]
    sec = sectors.reindex(valid)

    # Market-cap weights as baseline
    mc_weights = mc / mc.sum()

    if method == "tilt":
        tilt_factor = np.exp(Config.TILT_LAMBDA * comp)
        raw_weights = mc_weights * tilt_factor
    elif method == "score":
        raw_weights = comp.clip(lower=0)
    elif method == "top_n":
        top_rics = comp.nlargest(Config.TOP_N).index
        raw_weights = mc_weights.reindex(top_rics).fillna(0)
    else:
        raise ValueError(f"Unknown weighting method: {method}")

    # Renormalize
    weights = raw_weights / raw_weights.sum()

    # Single-name caps
    weights = weights.clip(upper=Config.MAX_WEIGHT)
    weights = weights[weights >= Config.MIN_WEIGHT]
    weights = weights / weights.sum()

    # Sector caps (max = benchmark sector weight × multiplier)
    if Config.SECTOR_CAP_MULTIPLE is not None:
        benchmark_sector_wt = mc.groupby(sec).sum() / mc.sum()
        sector_caps = benchmark_sector_wt * Config.SECTOR_CAP_MULTIPLE

        for sector in sec.unique():
            if pd.isna(sector):
                continue
            mask = sec == sector
            sector_wt = weights[mask].sum()
            cap = sector_caps.get(sector, 1.0)
            if sector_wt > cap and sector_wt > 0:
                scale = cap / sector_wt
                weights[mask] *= scale

        weights = weights / weights.sum()

    return weights

In [ ]:
mktcap = df_fy0["Company Market Cap"]
sectors = df_fy0["TRBC Economic Sector Name"]

weights = compute_index_weights(composite, mktcap, sectors)
print(f"Portfolio: {len(weights)} holdings")
print(f"Max weight: {weights.max():.2%}")
print(f"Min weight: {weights.min():.2%}")
print(f"Weight sum:  {weights.sum():.6f}")

### Top Holdings and Sector Allocation

In [ ]:
# Top 15 positions
top_holdings = weights.nlargest(15)
print("Top 15 Holdings:")
for ric, wt in top_holdings.items():
    sec = sectors.get(ric, "Unknown")
    print(f"  {ric:<12}  {wt:.2%}   {sec}")

In [ ]:
# Sector allocation — factor index vs benchmark
factor_sec = weights.groupby(sectors.reindex(weights.index)).sum().sort_values(ascending=False)
bench_sec = (mktcap / mktcap.sum()).groupby(sectors).sum().reindex(factor_sec.index)

sector_comparison = pd.DataFrame({
    "Factor Index": factor_sec,
    "Benchmark (R1K)": bench_sec,
    "Active Weight": factor_sec - bench_sec,
}).dropna()

print("Sector Allocation Comparison:")
print(sector_comparison.applymap(lambda x: f"{x:+.2%}" if isinstance(x, float) else x))

In [ ]:
# Visualize sector tilts
fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(sector_comparison))
width = 0.35

ax.bar(x - width/2, sector_comparison["Factor Index"] * 100, width, label="Factor Index", color="#2563eb")
ax.bar(x + width/2, sector_comparison["Benchmark (R1K)"] * 100, width, label="Russell 1000", color="#6b7280")

ax.set_xlabel("Sector")
ax.set_ylabel("Weight (%)")
ax.set_title("Sector Allocation: Factor Index vs Russell 1000")
ax.set_xticks(x)
ax.set_xticklabels(sector_comparison.index, rotation=45, ha="right")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

## 9. Backtest

The backtest engine handles:

- **Rebalancing to target weights** at specified dates
- **Transaction cost drag** on each rebalance (controlled by `COST_PER_SIDE_BPS`)
- **Buy-and-hold drift** between rebalances
- **Benchmark comparison** for excess return tracking

### Transaction Cost Model

$$TC_{rebal} = \frac{1}{2} \sum_i |w_{i,new} - w_{i,old}| \cdot c$$

where $c$ is the cost per side in basis points. The sum of absolute weight changes equals 2× one-way turnover, so we divide by 2 to get one-way turnover and multiply by cost per side.

### ⚠️ Single-Period Note

The cell below builds a `weights_history` dict with **one entry** (today's date). To do a proper multi-period backtest, you need to:

1. Build a loop over quarterly rebalance dates
2. Call `get_constituents(as_of_date=...)` for each date
3. Fetch point-in-time fundamentals for each date
4. Compute signals and weights for each date
5. Populate `weights_history` with one entry per quarter

The infrastructure is all here — `run_backtest()` already handles multi-period `weights_history`. The loop is the main thing to flesh out for a production backtest.

In [ ]:
def estimate_transaction_costs(
    old_weights: pd.Series, new_weights: pd.Series
) -> float:
    """Estimate transaction cost drag for a single rebalance."""
    all_rics = old_weights.index.union(new_weights.index)
    old = old_weights.reindex(all_rics, fill_value=0)
    new = new_weights.reindex(all_rics, fill_value=0)
    one_way_turnover = (new - old).abs().sum() / 2
    cost = one_way_turnover * (Config.COST_PER_SIDE_BPS / 10_000)
    return cost


def fetch_benchmark_returns(benchmark_ric: str, start: str, end: str) -> pd.Series:
    """Retrieve daily benchmark returns."""
    df = ld.get_history(
        universe=benchmark_ric,
        fields=["TRDPRC_1"],
        interval="1D",
        start=start,
        end=end,
    )
    prices = df["TRDPRC_1"].dropna()
    returns = prices.pct_change().dropna()
    returns.name = "benchmark"
    return returns


def fetch_price_history(rics: list, start: str, end: str) -> pd.DataFrame:
    """Retrieve daily closing prices for constituents."""
    prices = ld.get_history(
        universe=rics,
        fields=["TRDPRC_1"],
        interval="1D",
        start=start,
        end=end,
    )
    if isinstance(prices.columns, pd.MultiIndex):
        prices = prices["TRDPRC_1"]
    return prices


def run_backtest(
    weights_history: dict,
    price_df: pd.DataFrame,
    benchmark_returns: pd.Series,
) -> pd.DataFrame:
    """Simulate the factor index using historical weights and prices."""
    rebal_dates = sorted(weights_history.keys())
    all_dates = price_df.index.sort_values()

    portfolio_returns = []
    prev_weights = pd.Series(dtype=float)

    for i, rebal_date in enumerate(rebal_dates):
        target_weights = weights_history[rebal_date]

        tc = estimate_transaction_costs(prev_weights, target_weights)

        start_idx = all_dates.get_indexer([pd.Timestamp(rebal_date)], method="ffill")[0]
        if i + 1 < len(rebal_dates):
            end_idx = all_dates.get_indexer(
                [pd.Timestamp(rebal_dates[i + 1])], method="ffill"
            )[0]
        else:
            end_idx = len(all_dates)

        period_dates = all_dates[start_idx:end_idx]
        if len(period_dates) < 2:
            continue

        held_rics = target_weights.index.intersection(price_df.columns)
        period_prices = price_df.loc[period_dates, held_rics]
        daily_returns = period_prices.pct_change()

        w = target_weights.reindex(held_rics, fill_value=0)
        w = w / w.sum()

        for j, date in enumerate(period_dates[1:], 1):
            day_ret = daily_returns.loc[date]
            port_ret = (w * day_ret.reindex(w.index, fill_value=0)).sum()

            if j == 1:
                port_ret -= tc  # Deduct TC on rebalance day

            portfolio_returns.append({"date": date, "factor_return": port_ret})

            # Drift weights between rebalances
            w = w * (1 + day_ret.reindex(w.index, fill_value=0))
            w_sum = w.sum()
            if w_sum > 0:
                w = w / w_sum

        prev_weights = target_weights

    ret_df = pd.DataFrame(portfolio_returns).set_index("date")
    ret_df.index = pd.DatetimeIndex(ret_df.index)

    bmk = benchmark_returns.reindex(ret_df.index).fillna(0)

    results = pd.DataFrame({
        "factor_index": (1 + ret_df["factor_return"]).cumprod(),
        "benchmark": (1 + bmk).cumprod(),
    })
    results["excess"] = results["factor_index"] / results["benchmark"]
    return results

In [ ]:
# Define date range
end_date = datetime.now().strftime("%Y-%m-%d")
start_date = (datetime.now() - timedelta(days=365 * Config.LOOKBACK_YEARS)).strftime("%Y-%m-%d")

log.info(f"Backtest window: {start_date} to {end_date}")

# Fetch benchmark returns
benchmark_returns = fetch_benchmark_returns(Config.BENCHMARK_RIC, start_date, end_date)
print(f"Benchmark observations: {len(benchmark_returns)}")

In [ ]:
# Fetch prices for held constituents
log.info("Fetching price history for held stocks...")
held_rics = weights.index.tolist()
price_df = fetch_price_history(held_rics, start_date, end_date)
print(f"Price history shape: {price_df.shape}")

In [ ]:
# Build weights_history — single period for this demo
today_str = datetime.now().strftime("%Y-%m-%d")
weights_history = {today_str: weights}

# Run backtest
log.info("Running backtest...")
results = run_backtest(weights_history, price_df, benchmark_returns)
results.tail()

## 10. Performance Analytics

Standard metrics to evaluate the factor index against the benchmark:

- **Annualized Return** — $(1 + \bar{r}_{daily})^{252} - 1$
- **Annualized Volatility** — $\sigma_{daily} \cdot \sqrt{252}$
- **Sharpe Ratio** — $\frac{\text{Ann. Return}}{\text{Ann. Vol}}$ (assuming zero risk-free rate; adjust for a proper Sharpe)
- **Max Drawdown** — largest peak-to-trough decline
- **Total Return** — cumulative return over the backtest window

In [ ]:
def compute_performance_stats(results: pd.DataFrame) -> dict:
    """Compute key performance metrics."""
    factor_ret = results["factor_index"].pct_change().dropna()
    bmk_ret = results["benchmark"].pct_change().dropna()
    excess_ret = factor_ret - bmk_ret

    trading_days = 252

    stats = {}
    for name, rets in [("Factor Index", factor_ret), ("Benchmark", bmk_ret), ("Excess", excess_ret)]:
        ann_ret = (1 + rets.mean()) ** trading_days - 1
        ann_vol = rets.std() * np.sqrt(trading_days)
        sharpe = ann_ret / ann_vol if ann_vol > 0 else 0
        cum = (1 + rets).cumprod()
        drawdown = cum / cum.cummax() - 1
        max_dd = drawdown.min()

        stats[name] = {
            "Annualized Return": f"{ann_ret:.2%}",
            "Annualized Volatility": f"{ann_vol:.2%}",
            "Sharpe Ratio": f"{sharpe:.2f}",
            "Max Drawdown": f"{max_dd:.2%}",
            "Total Return": f"{cum.iloc[-1] - 1:.2%}" if len(cum) > 0 else "N/A",
        }

    return stats


stats = compute_performance_stats(results)
stats_df = pd.DataFrame(stats)
print("Performance Summary:")
print(stats_df)

In [ ]:
def compute_turnover_stats(weights_history: dict) -> dict:
    """Compute turnover statistics across rebalance periods."""
    dates = sorted(weights_history.keys())
    turnovers = []

    for i in range(1, len(dates)):
        old_w = weights_history[dates[i - 1]]
        new_w = weights_history[dates[i]]
        all_rics = old_w.index.union(new_w.index)
        turnover = (
            new_w.reindex(all_rics, fill_value=0)
            - old_w.reindex(all_rics, fill_value=0)
        ).abs().sum() / 2
        turnovers.append(turnover)

    return {
        "Mean One-Way Turnover": f"{np.mean(turnovers):.2%}" if turnovers else "N/A",
        "Max One-Way Turnover": f"{np.max(turnovers):.2%}" if turnovers else "N/A",
        "Min One-Way Turnover": f"{np.min(turnovers):.2%}" if turnovers else "N/A",
        "Rebalance Count": len(dates),
        "Est. Annual TC Drag": (
            f"{np.mean(turnovers) * 4 * Config.COST_PER_SIDE_BPS / 10_000:.4%}"
            if turnovers else "N/A (single period)"
        ),
    }


turnover = compute_turnover_stats(weights_history)
print("Turnover Statistics:")
for k, v in turnover.items():
    print(f"  {k:<30} {v}")

## 11. Visualization

Two-panel chart showing cumulative returns and relative performance (factor index / benchmark).

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), height_ratios=[2, 1])
fig.suptitle("Shareholder Yield Factor Index vs Russell 1000", fontsize=14, fontweight="bold")

# Panel 1: Cumulative returns
ax1.plot(results.index, results["factor_index"], label="SY Factor Index", linewidth=1.5, color="#2563eb")
ax1.plot(results.index, results["benchmark"], label="Russell 1000", linewidth=1.5, color="#6b7280", linestyle="--")
ax1.set_ylabel("Growth of $1")
ax1.legend(loc="upper left")
ax1.grid(True, alpha=0.3)

# Panel 2: Relative performance
ax2.plot(results.index, results["excess"], label="Factor / Benchmark", linewidth=1.5, color="#059669")
ax2.axhline(y=1.0, color="#6b7280", linestyle=":", linewidth=0.8)
ax2.set_ylabel("Relative Performance")
ax2.set_xlabel("Date")
ax2.legend(loc="upper left")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Drawdown Chart

A separate drawdown panel is often more informative than looking at the raw return line. Sharp drawdowns in the factor index that don't show up in the benchmark are a warning sign that the factor is picking up idiosyncratic risk rather than a robust premium.

In [ ]:
factor_cum = results["factor_index"]
bench_cum = results["benchmark"]

factor_dd = factor_cum / factor_cum.cummax() - 1
bench_dd = bench_cum / bench_cum.cummax() - 1

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(results.index, factor_dd * 100, 0, color="#2563eb", alpha=0.4, label="SY Factor Index")
ax.fill_between(results.index, bench_dd * 100, 0, color="#6b7280", alpha=0.4, label="Russell 1000")
ax.set_ylabel("Drawdown (%)")
ax.set_xlabel("Date")
ax.set_title("Drawdown Comparison")
ax.legend(loc="lower left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Close Session

Good practice to close the session when done, especially on long-running kernels.

In [ ]:
ld.close_session()
log.info("LSEG session closed.")

## Next Steps

Things worth extending once the single-period pipeline is working:

**Multi-period backtest.** Wrap Sections 4–8 in a loop over quarterly rebalance dates, calling `get_constituents(as_of_date=...)` and fetching point-in-time fundamentals for each date. This is the biggest lift but essential for honest backtest results.

**Point-in-time fundamentals.** Cash flow data is reported with a lag — typically 45–90 days after quarter-end. When you're rebalancing on, say, June 30th, you should only be using fundamental data that was actually available on that date. Use the `FPeriod` parameter in LSEG with explicit date offsets to enforce this.

**Signal diagnostics with alphalens-reloaded.** Before investing a lot of time in the full methodology, feed the composite score through alphalens-reloaded to see if it actually predicts forward returns. Quantile spreads and rolling IC are the key metrics.

**Quality filter.** Add a payout-ratio screen to avoid yield traps — e.g., exclude stocks with payout ratios above 100% of earnings, or with negative free cash flow.

**Debt paydown in shareholder yield.** The Mebane Faber definition includes net debt reduction. Adding `TR.F.LTDebtRepaid` and `TR.F.LTDebtIssued` to the data fetch and including `max(debt_repaid - debt_issued, 0)` in the numerator gives you the full "total capital allocation" signal.